# Instance creation stats

This script generate a summary of how many Instance records were created by each staff user from a user-supplied number of days in the past until today.

## 1. Environment setup

In [ ]:
# This script will import the following Python libraries, but not install them. If you are missing any of these, you can install them via the command line like this:
# !pip install pandas
import pandas as pd
import requests
from datetime import datetime, timedelta   

pd.set_option('display.max_columns', None)

## 2. Login

In [ ]:

%run folio_auth.ipynb

## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. 

In [ ]:
def fetch_all_records(endpoint, records_key, limit=1000):
    """
    Fetch all records from a paginated FOLIO endpoint.

    endpoint: path like "/accounts", "/groups", "/users"
    records_key: the JSON key holding the list of records, e.g. "accounts", "usergroups", "users"
    """
    all_records = []
    offset = 0
    base_url = OKAPI_URL.copy()
    headers = HEADERS.copy()


    while True:
        response = requests.get(
            f"{base_url}{endpoint}",
            headers=headers,
            params={"limit": limit, "offset": offset},
        )
        response.raise_for_status()  # fail loudly and clearly if something's wrong
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break  # last page
        offset += limit

    return all_records

## 4. Get # of Days to Search from Input
The number entered will be used to search for Instance records with a date created of {number of days} - today's date. You can choose to bypass this and use a standard calculation if you prefer not to have an input.


In [ ]:

while True:
    try:
        from_days = int(input("Enter the number of days to search back from today, or press Enter to use the default of 90 days: ") or 90)
        if from_days < 0:  
            print("Number cannot be negative")
            continue
        else:                
            date_x_days_ago = datetime.now() - timedelta(days=from_days)
            search_date =str(date_x_days_ago.strftime("%Y-%m-%d"))
            print("You entered:", from_days, "days. The search will look for Instance records created since: ", search_date)
            break
    except ValueError:
        print("Please enter a valid number.")

## 4. Pull data from each endpoint

In [ ]:

def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records


staff_raw = fetch_all_records(
    "/users",
    records_key="users",
    query='type=="staff"',
) 


instances_raw = fetch_all_records(
    "/instance-storage/instances",
    records_key="instances",
    query='metadata.createdDate >= ' + search_date,
)

print(f"staff:     {len(staff_raw)}")
print(f"instances: {len(instances_raw)}")


### Inspect the outputted data

In [ ]:
# Inspect staff records
staff_df    = pd.DataFrame(staff_raw)
staff_df.head()

In [ ]:
#Inspect instance records
instances_df   = pd.DataFrame(instances_raw)
instances_df.head()

## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?


In [ ]:
print(staff_df.columns.tolist())
print(instances_df.columns.tolist())

In [ ]:
# Spot-check types of the columns you intend to join on
# Since the user ID in the instance records is nested in the metadata dictionary, we need to extract it first
print(staff_df['id'].dtype)
print(instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None).dtype)

## 6. Merge

Join the Instances file with the Users file - matching on the UUID of the person who created the Instance record. 


Then inspect the first few rows to spot check that the merge worked correctly.

In [ ]:
instances_users = instances_df.merge(staff_df, left_on=instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None), right_on='id', how='left', suffixes=('_instance', '_user')
)
instances_users.head()

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [ ]:
print("Original Instance rows:", len(instances_df))
print("After merging with staff:  ", len(instances_users))

## 8. Analyze the combined dataset

Now that instances and users are joined, you can ask questions using pands functions.
Below is a summary of the number of instances created by user. 

In [ ]:
summary=instances_users.groupby('username')['id_instance'].count().sort_values(ascending=False)
print("Summary of instance records created by staff from", search_date, "to today:")
print(summary)

## 9. Simplify Your Data
A combined dataframe will include many columns that you won't want. You can create a new dataframe by extracting only the columns you want into a new set. 

In [ ]:
flattened_stf = pd.DataFrame()
flattened_stf['username'] = staff_df['username']
flattened_stf['firstName'] = staff_df['personal'].apply(lambda x: x.get('firstName') if isinstance(x, dict) else None)
flattened_stf['lastName'] = staff_df['personal'].apply(lambda x: x.get('lastName') if isinstance(x, dict) else None)
flattened_stf['staffId']= staff_df['id']
print("Basic staff information:")
flattened_stf.head()


In [ ]:
flattened_inst = pd.DataFrame()
flattened_inst['hrid'] = instances_df['hrid']
flattened_inst['title'] = instances_df['title']
flattened_inst['createdByUserId'] = instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x,dict) else None)
flattened_inst['createdDate']=pd.to_datetime(instances_df['metadata'].apply(lambda x: x.get('createdDate') if isinstance(x,dict) else None))

print("Basic Instance information")
print(flattened_inst)

## 10. Inspect and join the new, simpler dataframes

In [ ]:
print(flattened_stf.columns.tolist())
print(flattened_inst.columns.tolist())

In [ ]:
# Spot-check types of the columns you intend to join on
# Since the user ID in the instance records is nested in the metadata dictionary, we need to extract it first
print(flattened_stf['staffId'].dtype)
print(flattened_inst['createdByUserId'].dtype)

In [ ]:
stats_df = flattened_inst.merge(flattened_stf, left_on=flattened_inst['createdByUserId'], right_on='staffId', how='left', suffixes=('_instance', '_user')
)
stats_df.head()

## 11. Write to a File
Output your data to work on further, if necessary

In [ ]:
stats_df.to_csv('Instances Created since ' + str(search_date) + '.csv', index=False)